In [4]:
# IMPORTS
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.svm import SVC
from sklearn.model_selection import PredefinedSplit, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import confusion_matrix, classification_report


In [5]:
# Load modelling dataset
DATA_PATH = Path("C:/Users/janku/Documents/KCL/Research Project/Research Project/data/processed")
df_model = pd.read_csv(DATA_PATH / "androids_model_dataset_basic.csv")

# Feature columns
feature_names = [f"mfcc_{i+1}" for i in range(13)] + ["pitch_mean", "energy_mean"]

# Clean fold labels
df_model["fold"] = df_model["fold"].astype(str).str.strip()
df_model["speech_type"] = df_model["speech_type"].astype(str).str.strip().str.lower()

fold_map = {"fold1": 0, "fold2": 1, "fold3": 2, "fold4": 3, "fold5": 4}

# SVM pipeline
svm_model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", probability=True, random_state=42))
])

# Helper function
def run_svm_analysis(df_subset, label):
    df_subset = df_subset.dropna(subset=feature_names + ["depressed", "fold"]).copy()
    df_subset = df_subset[df_subset["fold"].isin(fold_map.keys())].copy()

    X = df_subset[feature_names].values
    y = df_subset["depressed"].astype(int).values
    test_fold = df_subset["fold"].map(fold_map).values
    ps = PredefinedSplit(test_fold=test_fold)

    scores = cross_validate(
        svm_model,
        X,
        y,
        cv=ps,
        scoring=["accuracy", "f1", "roc_auc"],
        return_train_score=False,
        error_score="raise"
    )

    return {
        "subset": label,
        "n_rows": len(df_subset),
        "n_depressed": int(df_subset["depressed"].sum()),
        "n_control": int((1 - df_subset["depressed"]).sum()),
        "accuracy_mean": scores["test_accuracy"].mean(),
        "accuracy_std": scores["test_accuracy"].std(),
        "f1_mean": scores["test_f1"].mean(),
        "f1_std": scores["test_f1"].std(),
        "roc_auc_mean": scores["test_roc_auc"].mean(),
        "roc_auc_std": scores["test_roc_auc"].std(),
    }

# Run analyses
results = []

results.append(run_svm_analysis(df_model, "all"))
results.append(run_svm_analysis(df_model[df_model["speech_type"] == "interview"], "interview"))
results.append(run_svm_analysis(df_model[df_model["speech_type"] == "read"], "read"))

results_df = pd.DataFrame(results)
print(results_df)

# Save results
RESULTS_PATH = Path("C:/Users/janku/Documents/KCL/Research Project/Research Project/results/metrics")

results_df.to_csv(RESULTS_PATH / "androids_svm_interview_vs_reading_results.csv", index=False)

      subset  n_rows  n_depressed  n_control  accuracy_mean  accuracy_std  \
0        all     224          122        102       0.879643      0.053151   
1  interview     115           64         51       0.653327      0.116435   
2       read     109           58         51       0.654466      0.094339   

    f1_mean    f1_std  roc_auc_mean  roc_auc_std  
0  0.887454  0.037462      0.934903     0.026798  
1  0.698703  0.097652      0.765099     0.105916  
2  0.665170  0.116495      0.798328     0.106823  


In [ ]:
# Load modelling dataset
DATA_PATH = Path("C:/Users/janku/Documents/KCL/Research Project/Research Project/data/processed")
df_model = pd.read_csv(DATA_PATH / "androids_model_dataset_basic.csv")

# Feature columns
feature_names = [f"mfcc_{i+1}" for i in range(13)] + ["pitch_mean", "energy_mean"]

# Clean fold labels
df_model["fold"] = df_model["fold"].astype(str).str.strip()
df_model["speech_type"] = df_model["speech_type"].astype(str).str.strip().str.lower()

fold_map = {"fold1": 0, "fold2": 1, "fold3": 2, "fold4": 3, "fold5": 4}

# SVM pipeline
svm_model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", probability=True, random_state=42))
])

# Helper function
def run_svm_analysis(df_subset, label):
    df_subset = df_subset.dropna(subset=feature_names + ["depressed", "fold"]).copy()
    df_subset = df_subset[df_subset["fold"].isin(fold_map.keys())].copy()

    X = df_subset[feature_names].values
    y = df_subset["depressed"].astype(int).values
    test_fold = df_subset["fold"].map(fold_map).values
    ps = PredefinedSplit(test_fold=test_fold)

    scores = cross_validate(
        svm_model,
        X,
        y,
        cv=ps,
        scoring=["accuracy", "f1", "roc_auc"],
        return_train_score=False,
        error_score="raise"
    )

    return {
        "subset": label,
        "n_rows": len(df_subset),
        "n_depressed": int(df_subset["depressed"].sum()),
        "n_control": int((1 - df_subset["depressed"]).sum()),
        "accuracy_mean": scores["test_accuracy"].mean(),
        "accuracy_std": scores["test_accuracy"].std(),
        "f1_mean": scores["test_f1"].mean(),
        "f1_std": scores["test_f1"].std(),
        "roc_auc_mean": scores["test_roc_auc"].mean(),
        "roc_auc_std": scores["test_roc_auc"].std(),
    }

# Run analyses
results = []

results.append(run_svm_analysis(df_model, "all"))
results.append(run_svm_analysis(df_model[df_model["speech_type"] == "interview"], "interview"))
results.append(run_svm_analysis(df_model[df_model["speech_type"] == "read"], "read"))

results_df = pd.DataFrame(results)
print(results_df)

# Save results
RESULTS_PATH = Path("C:/Users/janku/Documents/KCL/Research Project/Research Project/results/metrics")

results_df.to_csv(RESULTS_PATH / "androids_svm_interview_vs_reading_results.csv", index=False)